Compute DEGURBA at the centroid of every row's geometry. 

In [ ]:
import geopandas as gpd
import pandas as pd
from rasterstats import point_query
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling
from pathlib import Path
from datetime import datetime


DEGURBA_CATEGORIES = {
    130: 'Urban_Centre',
    223: 'Dense_Urban_Cluster',
    222: 'Semi_dense_Urban_Cluster',
    221: 'Suburban_Peri_urban',
    313: 'Rural_Cluster',
    312: 'Low_Density_Rural',
    311: 'Very_Low_Density_Rural',
    310: 'Water'
}

DEGURBA_RASTER      = r"C:\Users\agz90fk\Documents\EO4CAM\3_Daten\Input\DGURBA\DGURBA_LEVEL2_GRD_2021\DGUR_LEVEL2_GRD_1KM_2021.tif"
DEGURBA_REPROJECTED = r"C:\Users\agz90fk\Documents\EO4CAM\3_Daten\Output\Masterarbeit\Comparison_NHDA_RA_v2\DEGURBA_reprojected_25832.tif"
INPUT_FILE          = r"C:\Users\agz90fk\Documents\EO4CAM\3_Daten\Output\Masterarbeit\Comparison_NHDA_RA_v2\Comparison_LST_NDVI.gpkg"
OUTPUT_FILE         = r"C:\Users\agz90fk\Documents\EO4CAM\3_Daten\Output\Masterarbeit\Comparison_NHDA_RA_v2\Comparison_LST_NDVI_DEGURB.gpkg"


def calculate_degurba_for_polygon(polygon_geom, raster_path, verbose=False):
    try:
        centroid = polygon_geom.centroid
        result = point_query(
            [centroid],
            str(raster_path),
            interpolate='nearest',
            nodata=0
        )
        if not result or result[0] is None:
            return None, None
        category_code = int(result[0])
        category_name = DEGURBA_CATEGORIES.get(category_code)
        if category_name is None:
            return None, None
        return category_code, category_name
    except Exception as e:
        if verbose:
            print(f"      ⚠️  Error: {e}")
        return None, None


# ── File checks ────────────────────────────────────────────────────────────
print("\n" + "="*80)
print("DEGURBA LEVEL 2 DATA EXTRACTION")
print("="*80)

for label, path in [("DEGURBA raster", DEGURBA_RASTER), ("Input file", INPUT_FILE)]:
    exists = Path(path).exists()
    print(f"  {'✓' if exists else '❌'} {label}: {Path(path).name}")
    if not exists:
        raise FileNotFoundError(path)

start_time = datetime.now()

# ── Load ───────────────────────────────────────────────────────────────────
print("\nLoading input file...")
gdf = gpd.read_file(INPUT_FILE)
print(f"  → {len(gdf)} total rows, {gdf['nhda_id'].nunique()} unique nhda_ids")
print(f"  → type values: {gdf['type'].unique()}")
print(f"  → CRS: {gdf.crs}")

# ── Reproject raster if needed ─────────────────────────────────────────────
print("\nChecking raster CRS...")
with rasterio.open(DEGURBA_RASTER) as src:
    if src.crs == gdf.crs:
        print("  ✓ CRS match — no reprojection needed")
        raster_path = Path(DEGURBA_RASTER)
    else:
        print(f"  ⚠️  CRS mismatch — reprojecting to {gdf.crs}...")
        reproj_path = Path(DEGURBA_REPROJECTED)
        reproj_path.parent.mkdir(parents=True, exist_ok=True)
        transform, width, height = calculate_default_transform(
            src.crs, gdf.crs, src.width, src.height, *src.bounds
        )
        kwargs = src.meta.copy()
        kwargs.update({'crs': gdf.crs, 'transform': transform, 'width': width, 'height': height})
        with rasterio.open(reproj_path, 'w', **kwargs) as dst:
            for i in range(1, src.count + 1):
                reproject(
                    source=rasterio.band(src, i),
                    destination=rasterio.band(dst, i),
                    src_transform=src.transform,
                    src_crs=src.crs,
                    dst_transform=transform,
                    dst_crs=gdf.crs,
                    resampling=Resampling.nearest
                )
        print(f"  ✓ Reprojected and saved to: {reproj_path}")
        raster_path = reproj_path

# ── Step 1: Compute DEGURBA for every row from its own geometry ────────────
print(f"\nStep 1: Computing DEGURBA for all {len(gdf)} rows...")
gdf = gdf.reset_index(drop=True)
gdf['degurba_code']  = None
gdf['degurba_label'] = None

for idx, row in gdf.iterrows():
    if idx % 100 == 0:
        print(f"  {idx+1}/{len(gdf)}...")
    code, label = calculate_degurba_for_polygon(row.geometry, raster_path)
    gdf.at[idx, 'degurba_code']  = code
    gdf.at[idx, 'degurba_label'] = label

print(f"  ✓ Filled: {gdf['degurba_code'].notna().sum()} / {len(gdf)}")

# ── Step 2: Build lookup tables (nhda_id → degurba) for each type ──────────
print("\nStep 2: Adding partner DEGURBA values...")

nhda_lookup = (
    gdf[gdf['type'] == 'NHDA'][['nhda_id', 'degurba_code', 'degurba_label']]
    .rename(columns={'degurba_code': 'nhda_degurba_code', 'degurba_label': 'nhda_degurba_label'})
)
ra_lookup = (
    gdf[gdf['type'] == 'RA'][['nhda_id', 'degurba_code', 'degurba_label']]
    .rename(columns={'degurba_code': 'ra_degurba_code', 'degurba_label': 'ra_degurba_label'})
)

result = gdf.merge(nhda_lookup, on='nhda_id', how='left')
result = result.merge(ra_lookup,   on='nhda_id', how='left')
result = gpd.GeoDataFrame(result, crs=gdf.crs)

# ── Summary ────────────────────────────────────────────────────────────────
print(f"\n{'='*80}")
print("SUMMARY")
print(f"{'='*80}")
print(f"  Output rows: {len(result)}  (NHDA: {(result['type']=='NHDA').sum()}, RA: {(result['type']=='RA').sum()})")
print(f"  degurba filled (own):   {result['degurba_code'].notna().sum()} ({result['degurba_code'].notna().mean()*100:.1f}%)")
print(f"  nhda_degurba filled:    {result['nhda_degurba_code'].notna().sum()} ({result['nhda_degurba_code'].notna().mean()*100:.1f}%)")
print(f"  ra_degurba filled:      {result['ra_degurba_code'].notna().sum()} ({result['ra_degurba_code'].notna().mean()*100:.1f}%)")
print(f"\nNHDA rows — own DEGURBA distribution:")
print(result[result['type']=='NHDA']['degurba_label'].value_counts().to_string())
print(f"\nRA rows — own DEGURBA distribution:")
print(result[result['type']=='RA']['degurba_label'].value_counts().to_string())
print(f"\nRows where nhda_degurba and ra_degurba differ: "
      f"{(result['nhda_degurba_label'] != result['ra_degurba_label']).sum()}")

# ── Save ───────────────────────────────────────────────────────────────────
output_path = Path(OUTPUT_FILE)
output_path.parent.mkdir(parents=True, exist_ok=True)
result.to_file(OUTPUT_FILE, driver='GPKG')
print(f"\n✓ Done in {datetime.now() - start_time}")
print(f"  Output: {OUTPUT_FILE}  ({output_path.stat().st_size / 1024 / 1024:.1f} MB)")
print("="*80 + "\n")


In [ ]:
import geopandas as gpd
import pandas as pd

# =============================================================================
# INPUT
# =============================================================================

INPUT_GPKG = r"C:\Users\agz90fk\Documents\EO4CAM\3_Daten\Output\Masterarbeit\Comparison_NHDA_RA_v2\Comparison_LST_NDVI_DEGURB.gpkg"
LAYER = "Comparison_LST_NDVI_DEGURB"
# =============================================================================
# LOAD DATA
# =============================================================================

gdf = gpd.read_file(INPUT_GPKG, layer=LAYER)

# Nur NHDA-Zeilen verwenden
gdf = gdf[gdf["type"] == "NHDA"].copy()

print(f"Number of NHDAs: {len(gdf)}")
# =============================================================================
# COMPARE DEGURBA
# =============================================================================

gdf["same_degurba"] = (
    gdf["nhda_degurba_code"] == gdf["ra_degurba_code"]
)

summary = (
    gdf["same_degurba"]
    .value_counts()
    .rename(index={True: "Same", False: "Different"})
    .to_frame("Count")
)

summary["Percent"] = summary["Count"] / len(gdf) * 100

print("\nOverall comparison")
print(summary.round(2))

# =============================================================================
# DEGURBA TRANSITIONS (NHDA -> RA)
# =============================================================================

# Absolute Häufigkeiten
transition_counts = pd.crosstab(
    gdf["nhda_degurba_code"],
    gdf["ra_degurba_code"]
)

# Prozent innerhalb jeder NHDA-Klasse
transition_percent = pd.crosstab(
    gdf["nhda_degurba_code"],
    gdf["ra_degurba_code"],
    normalize="index"
) * 100

print("\n===================================================")
print("ABSOLUTE COUNTS (NHDA -> RA)")
print("===================================================")
print(transition_counts)

print("\n===================================================")
print("ROW PERCENTAGES (within each NHDA class)")
print("===================================================")
print(transition_percent.round(1))

# =============================================================================
# DETAILED SUMMARY
# =============================================================================

print("\n===================================================")
print("DETAILED SUMMARY")
print("===================================================")

for nhda_class in transition_counts.index:

    total = transition_counts.loc[nhda_class].sum()

    print(f"\nNHDA DEGURBA {nhda_class} (n={total})")

    summary = pd.DataFrame({
        "RA_Count": transition_counts.loc[nhda_class],
        "Percent": transition_percent.loc[nhda_class].round(1)
    })

    summary = summary[summary["RA_Count"] > 0]
    summary = summary.sort_values("RA_Count", ascending=False)

    print(summary)
